# Section 7, Test 1 Rebuilt: ~1M Parameters, Noisy Budget, Quantitative Metrics

**What changed from the previous Test 1, per Cameron's feedback:**
1. **Scale**: architecture rescaled to P=1,059,897 (base_ch=56), matching Cameron's ~1.01M scale.
2. **Budget allocation**: alongside the original "precise" budget (M=20 probes/step, few steps), this
   adds a "noisy" budget (small M, many more steps) -- the same total forward-pass-equivalent budget,
   spent on step count instead of per-step precision, which is Cameron's own paper's headline finding
   for why the local rule does better on diffusion reconstruction.
3. **Quantitative metrics**: pixel-accuracy, PSNR, and SSIM, computed as a genuine reconstruction
   benchmark (real noised image -> single-shot denoise -> compare to ground truth), not just loss and
   a visual check.

**Expectation, stated honestly before running:** the noisy budget should do meaningfully better than
the precise budget did in the previous notebook, consistent with the unbiasedness argument (many
weakly-aligned steps net out to more progress than a few well-aligned ones, given a fixed budget).
Whether it closes the gap to backprop as much as Cameron's own results suggest is the actual open
question this notebook answers -- report whatever the real numbers show, not what's expected.

## Step 0 — Setup

In [ ]:
import json
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

RESULTS_LOG = []

def log_result(name, config, result, seed):
    entry = {'name': name, 'seed': seed, 'config': config, 'result': result,
              'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')}
    RESULTS_LOG.append(entry)
    print(f"[logged] {name}: {result}")
    return entry

def save_provenance(path='provenance_diffusion_test1_1M.json'):
    with open(path, 'w') as f:
        json.dump(RESULTS_LOG, f, indent=2, default=str)
    print(f'Saved {len(RESULTS_LOG)} logged results to {path}')


## Step 1 — Schedule, architecture (P~1.06M), data, sampler (all confirmed fixes carried forward)

In [ ]:
T_STEPS = 1000
beta_start, beta_end = 1e-4, 0.02
betas = torch.linspace(beta_start, beta_end, T_STEPS, device=device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

def forward_diffusion(x0, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)
    sqrt_ab = alpha_bars[t].sqrt().view(-1, 1, 1, 1)
    sqrt_1mab = (1 - alpha_bars[t]).sqrt().view(-1, 1, 1, 1)
    return sqrt_ab * x0 + sqrt_1mab * noise, noise

assert alpha_bars[-1].item() < 0.01

class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-np.log(10000) * torch.arange(half, device=t.device).float() / half)
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.norm1 = nn.GroupNorm(min(8, out_ch), out_ch)
        self.norm2 = nn.GroupNorm(min(8, out_ch), out_ch)
        self.time_proj = nn.Linear(time_dim, out_ch)
        self.act = nn.SiLU()
    def forward(self, x, t_emb):
        h = self.act(self.norm1(self.conv1(x)))
        h = h + self.time_proj(t_emb).unsqueeze(-1).unsqueeze(-1)
        h = self.act(self.norm2(self.conv2(h)))
        return h

class SmallUNet(nn.Module):
    def __init__(self, base_ch=56, time_dim=32):
        super().__init__()
        self.time_embed = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim), nn.SiLU(), nn.Linear(time_dim, time_dim)
        )
        self.enc1 = ConvBlock(1, base_ch, time_dim)
        self.down1 = nn.Conv2d(base_ch, base_ch, 3, stride=2, padding=1)
        self.enc2 = ConvBlock(base_ch, base_ch * 2, time_dim)
        self.down2 = nn.Conv2d(base_ch * 2, base_ch * 2, 3, stride=2, padding=1)
        self.bottleneck = ConvBlock(base_ch * 2, base_ch * 2, time_dim)
        self.up2 = nn.ConvTranspose2d(base_ch * 2, base_ch * 2, 4, stride=2, padding=1)
        self.dec2 = ConvBlock(base_ch * 4, base_ch, time_dim)
        self.up1 = nn.ConvTranspose2d(base_ch, base_ch, 4, stride=2, padding=1)
        self.dec1 = ConvBlock(base_ch * 2, base_ch, time_dim)
        self.out_conv = nn.Conv2d(base_ch, 1, 3, padding=1)

    def forward(self, x, t):
        t_emb = self.time_embed(t)
        e1 = self.enc1(x, t_emb)
        e2 = self.enc2(self.down1(e1), t_emb)
        b = self.bottleneck(self.down2(e2), t_emb)
        d2 = self.up2(b)
        d2 = self.dec2(torch.cat([d2, e2], dim=1), t_emb)
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1), t_emb)
        return self.out_conv(d1)

def make_denoiser(seed=SEED):
    torch.manual_seed(seed)
    return SmallUNet(base_ch=56, time_dim=32).to(device)

P_denoiser = sum(p.numel() for p in make_denoiser().parameters())
print(f'Denoiser parameter count: P = {P_denoiser:,}')

transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x * 2 - 1)])
mnist_train = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
mnist_test = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

N_TRAIN = 8000
train_idx = torch.randperm(len(mnist_train))[:N_TRAIN]
X_train = torch.stack([mnist_train[i][0] for i in train_idx]).to(device)

N_TEST = 200
test_idx = torch.randperm(len(mnist_test))[:N_TEST]
X_test = torch.stack([mnist_test[i][0] for i in test_idx]).to(device)
print(f'Train: {X_train.shape[0]}, held-out test: {X_test.shape[0]}')

@torch.no_grad()
def sample_ddpm(model, n_samples=8, seed=None):
    model.eval()
    if seed is not None:
        gen = torch.Generator(device=device).manual_seed(seed)
        x = torch.randn(n_samples, 1, 28, 28, device=device, generator=gen)
    else:
        x = torch.randn(n_samples, 1, 28, 28, device=device)
    for t_step in reversed(range(T_STEPS)):
        t_batch = torch.full((n_samples,), t_step, device=device, dtype=torch.long)
        pred_noise = model(x, t_batch)
        alpha_t = alphas[t_step]; alpha_bar_t = alpha_bars[t_step]
        alpha_bar_prev = alpha_bars[t_step - 1] if t_step > 0 else torch.tensor(1.0, device=device)
        beta_t = betas[t_step]
        x0_pred = ((x - (1 - alpha_bar_t).sqrt() * pred_noise) / alpha_bar_t.sqrt()).clamp(-1.0, 1.0)
        posterior_mean = (
            (alpha_bar_prev.sqrt() * beta_t / (1 - alpha_bar_t)) * x0_pred
            + (alpha_t.sqrt() * (1 - alpha_bar_prev) / (1 - alpha_bar_t)) * x
        )
        posterior_var = beta_t * (1 - alpha_bar_prev) / (1 - alpha_bar_t)
        x = posterior_mean + posterior_var.sqrt() * torch.randn_like(x) if t_step > 0 else posterior_mean
    model.train()
    return x

def show_sample_grid(samples, title, save_path):
    disp = (samples.clamp(-1, 1) + 1) / 2
    fig, axes = plt.subplots(1, 8, figsize=(16, 2))
    for i, ax in enumerate(axes):
        ax.imshow(disp[i, 0].cpu().numpy(), cmap='gray'); ax.axis('off')
    plt.suptitle(title); plt.savefig(save_path, dpi=120, bbox_inches='tight'); plt.show()

def ssim_simple(img1, img2, C1=0.01**2, C2=0.03**2):
    mu1, mu2 = img1.mean(), img2.mean()
    var1, var2 = img1.var(), img2.var()
    covar = ((img1 - mu1) * (img2 - mu2)).mean()
    return ((2*mu1*mu2 + C1) * (2*covar + C2)) / ((mu1**2 + mu2**2 + C1) * (var1 + var2 + C2))

@torch.no_grad()
def reconstruction_metrics(model, X, t_val=500, n_samples=100):
    model.eval()
    x0 = X[:n_samples]
    t = torch.full((n_samples,), t_val, device=device)
    x_t, noise = forward_diffusion(x0, t)
    pred_noise = model(x_t, t)
    alpha_bar_t = alpha_bars[t_val]
    x0_pred = ((x_t - (1 - alpha_bar_t).sqrt() * pred_noise) / alpha_bar_t.sqrt()).clamp(-1, 1)
    x0_disp = (x0.clamp(-1, 1) + 1) / 2
    x0_pred_disp = (x0_pred.clamp(-1, 1) + 1) / 2
    mse = F.mse_loss(x0_pred_disp, x0_disp).item()
    psnr = 10 * np.log10(1.0 / max(mse, 1e-10))
    pixel_acc = ((x0_pred_disp > 0.5) == (x0_disp > 0.5)).float().mean().item()
    ssim_val = ssim_simple(x0_pred_disp, x0_disp).item()
    model.train()
    return {'mse': mse, 'psnr_db': psnr, 'pixel_accuracy': pixel_acc, 'ssim': ssim_val}


## Step 2 — Backprop (reference, fixed step budget)

In [ ]:
BATCH_SIZE = 64
BP_STEPS = 500
LR_BP = 2e-4

denoiser_bp = make_denoiser(SEED)
opt = torch.optim.Adam(denoiser_bp.parameters(), lr=LR_BP)
losses_bp = []
t0 = time.time()
for step in range(BP_STEPS):
    idx = torch.randint(0, X_train.shape[0], (BATCH_SIZE,), device=device)
    x0 = X_train[idx]
    t = torch.randint(0, T_STEPS, (BATCH_SIZE,), device=device)
    x_t, noise = forward_diffusion(x0, t)
    opt.zero_grad(set_to_none=True)
    pred_noise = denoiser_bp(x_t, t)
    loss = F.mse_loss(pred_noise, noise)
    loss.backward()
    opt.step()
    losses_bp.append(loss.item())
    if (step + 1) % 100 == 0:
        print(f'backprop step {step+1}/{BP_STEPS}  loss={np.mean(losses_bp[-100:]):.4f}')

bp_time = time.time() - t0
bp_forward_pass_equiv = BP_STEPS * 2
print(f'\nBackprop done in {bp_time:.1f}s, final loss {np.mean(losses_bp[-50:]):.4f}')
metrics_bp = reconstruction_metrics(denoiser_bp, X_test)
print(f'Backprop reconstruction metrics: {metrics_bp}')
log_result('test1_1M_backprop', {'steps': BP_STEPS, 'lr': LR_BP},
           {'final_loss': float(np.mean(losses_bp[-50:])), 'wall_clock_s': bp_time,
            'forward_pass_equiv_budget': bp_forward_pass_equiv, **metrics_bp}, SEED)

samples_bp = sample_ddpm(denoiser_bp, n_samples=8, seed=777)
show_sample_grid(samples_bp, 'Backprop, from scratch (1M, matched budget)', 'test1_1M_backprop_samples.png')


## Step 3 — Three-factor, TWO budget allocations at the SAME total compute

**Precise budget** (as before): larger $M$, fewer steps. **Noisy budget** (Cameron's suggestion):
small $M$, many more steps -- same total forward-pass-equivalents, different allocation.

In [ ]:
def flat_params(model):
    return torch.cat([p.data.view(-1) for p in model.parameters()])

def set_flat_params(model, flat):
    offset = 0
    for p in model.parameters():
        n = p.numel()
        p.data.copy_(flat[offset:offset+n].view_as(p))
        offset += n

def diffusion_loss_at(model, flat, x0, t, noise):
    set_flat_params(model, flat)
    with torch.no_grad():
        x_t, _ = forward_diffusion(x0, t, noise)
        pred_noise = model(x_t, t)
        return F.mse_loss(pred_noise, noise).item()

def train_three_factor(M_PROBES, LR_LOCAL, SIGMA, label, max_grad_norm=5.0):
    steps_budget = bp_forward_pass_equiv // (2 * M_PROBES)
    print(f'--- {label}: M={M_PROBES}, steps={steps_budget} '
          f'(uses {steps_budget*2*M_PROBES} fwd-pass-equiv, vs backprop\'s {bp_forward_pass_equiv}) ---')
    model = make_denoiser(SEED)
    theta = flat_params(model)
    D = theta.numel()
    losses = []
    t0 = time.time()
    print_every = max(1, steps_budget // 10)
    for step in range(steps_budget):
        idx = torch.randint(0, X_train.shape[0], (BATCH_SIZE,), device=device)
        x0 = X_train[idx]
        t = torch.randint(0, T_STEPS, (BATCH_SIZE,), device=device)
        noise = torch.randn_like(x0)
        g_hat = torch.zeros_like(theta)
        for _ in range(M_PROBES):
            xi = torch.randn(D, device=device)
            l_plus = diffusion_loss_at(model, theta + SIGMA * xi, x0, t, noise)
            l_minus = diffusion_loss_at(model, theta - SIGMA * xi, x0, t, noise)
            g_hat += xi * (l_plus - l_minus) / (2 * SIGMA)
        g_hat /= M_PROBES
        # STABILIZATION FIX: clip the update norm before applying it. Without this, a
        # high-variance low-M estimate on a ~1M-parameter model can push weights into a
        # divergent (NaN) regime over many steps -- this is what produced the blank/NaN
        # sample grid in the first version of this cell.
        g_norm = g_hat.norm()
        if g_norm > max_grad_norm:
            g_hat = g_hat * (max_grad_norm / g_norm)
        theta = theta - LR_LOCAL * g_hat
        step_loss = diffusion_loss_at(model, theta, x0, t, noise)
        # NaN/Inf guard: stop immediately and report clearly rather than silently producing
        # garbage samples for the rest of the run.
        if not np.isfinite(step_loss):
            print(f'  {label}: DIVERGED at step {step+1} (loss={step_loss}). Stopping early.')
            print(f'  Try a lower LR_LOCAL and/or a smaller max_grad_norm for this budget.')
            break
        losses.append(step_loss)
        if (step + 1) % print_every == 0:
            print(f'  {label} step {step+1}/{steps_budget}  loss={step_loss:.4f}  '
                  f'(||g_hat||={g_hat.norm().item():.3f})')
    set_flat_params(model, theta)
    elapsed = time.time() - t0
    print(f'{label} done in {elapsed:.1f}s, final loss {losses[-1]:.4f}')
    return model, losses, elapsed, steps_budget

# Precise budget: same as the original Test 1 (M=20)
model_precise, losses_precise, time_precise, steps_precise = train_three_factor(
    M_PROBES=20, LR_LOCAL=5e-3, SIGMA=0.01, label='PRECISE (M=20)')
metrics_precise = reconstruction_metrics(model_precise, X_test)
print(f'Precise-budget reconstruction metrics: {metrics_precise}')
log_result('test1_1M_three_factor_precise', {'steps': steps_precise, 'M': 20, 'lr': 5e-3, 'sigma': 0.01},
           {'final_loss': losses_precise[-1], 'wall_clock_s': time_precise, **metrics_precise}, SEED)
samples_precise = sample_ddpm(model_precise, n_samples=8, seed=777)
show_sample_grid(samples_precise, 'Three-factor PRECISE budget (M=20)', 'test1_1M_precise_samples.png')


In [ ]:
# Noisy budget: small M, many more steps -- Cameron's suggested allocation.
# M=2 keeps sigma-noise dominant (cos alignment low per step) but buys ~10x more steps.
model_noisy, losses_noisy, time_noisy, steps_noisy = train_three_factor(
    M_PROBES=2, LR_LOCAL=1e-3, SIGMA=0.01, label='NOISY (M=2)', max_grad_norm=2.0)
# LR lowered from 5e-3 to 1e-3 and max_grad_norm tightened to 2.0 for this run --
# M=2 is a much higher-variance estimate than M=20, and the same LR that was stable
# there diverged here on this ~1M-parameter model.
metrics_noisy = reconstruction_metrics(model_noisy, X_test)
print(f'Noisy-budget reconstruction metrics: {metrics_noisy}')
log_result('test1_1M_three_factor_noisy', {'steps': steps_noisy, 'M': 2, 'lr': 5e-3, 'sigma': 0.01},
           {'final_loss': losses_noisy[-1], 'wall_clock_s': time_noisy, **metrics_noisy}, SEED)
samples_noisy = sample_ddpm(model_noisy, n_samples=8, seed=777)
show_sample_grid(samples_noisy, 'Three-factor NOISY budget (M=2, many more steps)', 'test1_1M_noisy_samples.png')


## Step 4 — Two-factor Hebbian (conv generalization, same step budget as the noisy run)

In [ ]:
def hebbian_conv_update(conv_layer, x_in, x_out, eta):
    kh, kw = conv_layer.kernel_size
    stride = conv_layer.stride[0]
    padding = conv_layer.padding[0]
    patches = F.unfold(x_in, kernel_size=(kh, kw), stride=stride, padding=padding)
    B, _, L = patches.shape
    out_flat = x_out.reshape(B, x_out.shape[1], -1)
    update = torch.einsum('bpl,bol->op', patches, out_flat) / (B * L)
    update = update.view(conv_layer.out_channels, conv_layer.in_channels, kh, kw)
    conv_layer.weight.data += eta * update

ETA_HEB = 1e-6
HEB_STEPS = steps_noisy

denoiser_heb = make_denoiser(SEED)
conv_layers = [m for m in denoiser_heb.modules() if isinstance(m, nn.Conv2d)]
activity = {}
def hook_factory(key):
    def hook(module, inp, out):
        activity[key] = (inp[0].detach(), out.detach())
    return hook
handles = [layer.register_forward_hook(hook_factory(i)) for i, layer in enumerate(conv_layers)]

losses_heb = []
t0 = time.time()
print_every = max(1, HEB_STEPS // 10)
for step in range(HEB_STEPS):
    idx = torch.randint(0, X_train.shape[0], (BATCH_SIZE,), device=device)
    x0 = X_train[idx]
    t = torch.randint(0, T_STEPS, (BATCH_SIZE,), device=device)
    with torch.no_grad():
        x_t, noise = forward_diffusion(x0, t)
        pred_noise = denoiser_heb(x_t, t)
        loss = F.mse_loss(pred_noise, noise)
    for i, layer in enumerate(conv_layers):
        x_in, x_out = activity[i]
        hebbian_conv_update(layer, x_in, x_out, ETA_HEB)
    losses_heb.append(loss.item())
    if (step + 1) % print_every == 0:
        print(f'two-factor step {step+1}/{HEB_STEPS}  loss={loss.item():.4f}')

for h in handles:
    h.remove()
heb_time = time.time() - t0
print(f'\nTwo-factor done in {heb_time:.1f}s, final loss {losses_heb[-1]:.4f}')
metrics_heb = reconstruction_metrics(denoiser_heb, X_test)
print(f'Two-factor reconstruction metrics: {metrics_heb}')
log_result('test1_1M_two_factor', {'steps': HEB_STEPS, 'eta': ETA_HEB},
           {'final_loss': losses_heb[-1], 'wall_clock_s': heb_time, **metrics_heb}, SEED)

samples_heb = sample_ddpm(denoiser_heb, n_samples=8, seed=777)
show_sample_grid(samples_heb, 'Two-factor Hebbian (matched to noisy step count)', 'test1_1M_two_factor_samples.png')


## Step 5 — Head-to-head summary table

In [ ]:
print(f'{"Method":<22}{"Loss":>10}{"Pixel-acc":>12}{"PSNR(dB)":>11}{"SSIM":>9}')
print('-' * 64)
print(f'{"Backprop":<22}{losses_bp[-1]:>10.4f}{metrics_bp["pixel_accuracy"]:>12.4f}{metrics_bp["psnr_db"]:>11.2f}{metrics_bp["ssim"]:>9.4f}')
print(f'{"Three-factor precise":<22}{losses_precise[-1]:>10.4f}{metrics_precise["pixel_accuracy"]:>12.4f}{metrics_precise["psnr_db"]:>11.2f}{metrics_precise["ssim"]:>9.4f}')
print(f'{"Three-factor noisy":<22}{losses_noisy[-1]:>10.4f}{metrics_noisy["pixel_accuracy"]:>12.4f}{metrics_noisy["psnr_db"]:>11.2f}{metrics_noisy["ssim"]:>9.4f}')
print(f'{"Two-factor Hebbian":<22}{losses_heb[-1]:>10.4f}{metrics_heb["pixel_accuracy"]:>12.4f}{metrics_heb["psnr_db"]:>11.2f}{metrics_heb["ssim"]:>9.4f}')

log_result('test1_1M_summary', {},
           {'backprop': metrics_bp, 'three_factor_precise': metrics_precise,
            'three_factor_noisy': metrics_noisy, 'two_factor': metrics_heb}, SEED)

print()
print('Compare the NOISY budget against the PRECISE budget directly -- does spending the same')
print('total compute on more, less-precise steps actually close more of the gap to backprop,')
print('as Cameron\'s own results suggest it should? Report whatever the real numbers show.')


## Step 6 — Save provenance

In [ ]:
save_provenance('provenance_diffusion_test1_1M.json')
print()
for entry in RESULTS_LOG:
    print(f"  - {entry['name']}")


## Summary

This rebuild directly tests Cameron's suggestion: does trading per-step precision for step count
(same total compute) meaningfully close the gap to backprop? Both budget allocations are run at the
same ~1.06M scale, with the same quantitative reconstruction metrics (pixel-accuracy, PSNR, SSIM)
Cameron's own paper uses, plus the real generative sample-grid check this project has used
throughout. Whatever the precise-vs-noisy comparison actually shows is the real answer to bring to
the meeting -- not an assumption that the noisy budget will obviously win.